# Imports

In [ ]:
# Imports
import os
import time
import warnings
import pandas as pd

# Silence TensorFlow C++ logs
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore")

import tensorflow as tf
from tensorflow.keras.models import load_model

import sys
sys.path.append("..")

from exofilt.NN_inference import (
    configure_tensorflow,
    filter_df,
    crop_tracks_from_df,
    create_generator,
    predict_multiple_models,
    subset_by_multiple_thresholds,
    save_subsets_by_threshold,
)
    
configure_tensorflow()

# User input

In [ ]:
# Root folder of the analysis.
# This should contain the preprocessing and tracking outputs
# generated by the previous steps of the pipeline (ImageJ scripts).
path_general = "../test_data/"

# Folder with trained NN models (.keras files).
path_models = "../models/"

# Optional plotting of prediction results
plot_predictions = False

# Default suggested thresholds for saved predictions
thresholds = [0.25, 0.5, 0.75]

In [ ]:
# ---------------------------
# Internal paths (in general, should not be changed)
# ---------------------------

# Folder containing preprocessed full-field-of-view videos. 
# Generated by the previous steps of the pipeline (ImageJ scripts)
path_wholeFOV = os.path.join(path_general, "3_Preprocessed")

# CSV file containing all detected tracks
path_csv_alltracks = os.path.join(
    path_general, 
    "4_Tracking",
    "tracks_all.csv",
)

# Folder where cropped track videos will be saved
path_crops = os.path.join(
    path_general,
    "4_Tracking", 
    "crops_raw",
)

# Folder where NN prediction results will be stored
path_NN_preds = os.path.join(
    path_general, 
    "4_Tracking",
    "NN_predictions",
)

# ---------------------------
# Validate required input paths
# ---------------------------
required_paths = {
    "path_general (analysis root folder)": path_general,
    "path_models (folder with trained NN models)": path_models,
    "path_wholeFOV (preprocessed videos)": path_wholeFOV,
}

for name, path in required_paths.items():
    if not os.path.isdir(path):
        raise FileNotFoundError(
            f"Required directory not found:\n"
            f"  {name}: {path}\n"
            f"Please check that the path exists and is correctly specified."
        )

if not os.path.isfile(path_csv_alltracks):
    raise FileNotFoundError(
        f"Required file not found:\n"
        f"  path_csv_alltracks: {path_csv_alltracks}\n"
        f"This file should contain the tracking results (tracks_all.csv)."
    )

# ---------------------------
# Create output folders
# ---------------------------
for d in [path_crops, path_NN_preds]:
    os.makedirs(d, exist_ok=True)

# ---------------------------
# Load models
# ---------------------------
model_names = sorted(os.listdir(path_models))

# ---------------------------
# Define permissive filter
# ---------------------------
permissive_filter = {
    'TRACK_MEAN_QUALITY': (2.146026188903055, 11.048775000308925),
    'TRACK_MEAN_Q_IN': (0.7834928777773622, 11.522475879646468),
    'MAX_DISTANCE_TRAVELED': (-0.4433314491526025, 4.124844553281212),
    'TRACK_DISPLACEMENT': (-0.6560135823760674, 3.5895958307948135),
    'TRACK_MIN_SPEED': (-0.007889382852321675, 0.03944691426160837),
    'TRACK_MAX_SPEED': (-0.22930417791072735, 1.7303495087366054),
    'TRACK_MEAN_SPEED': (0.012963139198257628, 0.17896838171621926),
    'TRACK_MEDIAN_SPEED': (0.007040380811807627, 0.16233584415775446),
    'TRACK_STD_SPEED': (-0.028436678013124623, 0.2653926994220601),
    'TRACK_MEAN_Q_ENV': (-1.0342213940109963, 3.0923475080487433),
    'TRACK_MEAN_Q_IN_BEFORE': (-2.954086212884812, 8.809739158267066),
    'TRACK_MEAN_Q_IN_AFTER': (-3.792362220703605, 7.129223634326266),
}

# Preprocessing: permissive filter + cropping

In [ ]:
%%time

df_tracks = pd.read_csv(path_csv_alltracks)

df_filtered = filter_df(
    data = df_tracks,
    filters = permissive_filter,
    verbose = True,
)

crop_tracks_from_df(
    df_tracks = df_filtered,
    path_wholeFOV = path_wholeFOV,
    path_crops = path_crops,
    size_crop = 10,
    extra_padding = 18,
)

# Create generator

In [ ]:
%%time

generator, generator_dummy = create_generator(
    path_crops = path_crops,
    with_dummy = True,
)

# Warm-up prediction (speeds up real inference)
path_model = os.path.join(path_models, model_names[0])
model = load_model(path_model)

t = model.predict(generator_dummy, verbose=1)

# Run Neural Network predictions

In [ ]:
%%time

df_preds = predict_multiple_models(
    generator=generator,
    model_names=model_names,
    path_models=path_models,
    plot=plot_predictions
)

# Save predictions
path_summary_NN_preds = os.path.join(path_NN_preds, "summary_NN_preds.csv")
df_preds.to_csv(path_summary_NN_preds, index = False)

print(f"\nSaved summary predictions: {path_summary_NN_preds}")

# Threshold exploration + save selected crops

In [ ]:
# Load summary NN predictions (just stored on the previous step)
df_preds = pd.read_csv(os.path.join(path_NN_preds, "summary_NN_preds.csv"))

# Recommended: first threshold at 0.25
thresholds = [0.75]

list_df_selected = subset_by_multiple_thresholds(df_preds, thresholds)

In [ ]:
# Once the thresholds are decided, save csv files with the selected tracks
save_subsets_by_threshold(
    list_df_selected = list_df_selected,
    df_alltracks = df_tracks,
    path_save_subset = path_NN_preds,
)